# FRESCO Pipeline

This notebook executes the pipeline by reading a single configuration file:

- `config/pipeline.yaml`

Advantages:

- A single location to modify parameters (SAM3 / Nemotron / thresholds / paths)
- Reproducible for colleagues and reviewers
- Consistent behavior with the terminal (CLI execution)

> Before running: Install `pyyaml` (`pip install pyyaml`) and `dotenv`, then set your `NVIDIA_API_KEY` in a `.env` file within this directory (`export NVIDIA_API_KEY={your-nvidia-api-key}`).


In [9]:
from pathlib import Path
import os, sys, subprocess, shlex
import yaml

CONFIG_PATH = Path("pipeline.yaml")
REPO_ROOT = Path(".").resolve()

print("Repo:", REPO_ROOT)
print("Config:", CONFIG_PATH.resolve())

if not os.path.exists(".env"):
    with open(".env", "w") as f:
        f.writeline("export NVIDIA_API_KEY={your-nvidia-api-key}")


Repo: /datadrive/AutoPBR
Config: /datadrive/AutoPBR/pipeline.yaml


## Helper: run comandi con output live


In [15]:
from dotenv import load_dotenv
from huggingface_hub import notebook_login

load_dotenv()
notebook_login()

def run(cmd: str):
    print("\n▶", cmd)
    subprocess.run(shlex.split(cmd), check=True)

def q(x) -> str:
    return shlex.quote(str(x))

def args_to_cli(args: dict, ctx: dict) -> str:
    parts = []
    for k, v in args.items():
        kk = k.replace("_","-")
        flag = f"--{kk}"
        if isinstance(v, bool):
            if v:
                parts.append(flag)
            continue
        if isinstance(v, str):
            v = v.format(**ctx)
        parts.append(flag)
        parts.append(str(v))
    return " ".join(q(p) for p in parts)


## Load config + sanity checks


In [11]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing {CONFIG_PATH}. Create/commit it in the repo.")

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

photos_dir = cfg["paths"]["photos_dir"]
out_root   = cfg["paths"]["output_root"]
ctx = {"photos_dir": photos_dir, "output_root": out_root}

# Env check
missing = [k for k in cfg.get("env", {}).get("required", []) if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}. Set them before running (e.g., export NVIDIA_API_KEY=...).")
print("✅ Env OK")

# Input folder check
photos_path = REPO_ROOT / photos_dir
photos_path.mkdir(exist_ok=True, parents=True)
imgs = list(photos_path.glob("*.jpg")) + list(photos_path.glob("*.jpeg")) + list(photos_path.glob("*.png"))
print(f"📷 images in {photos_path}: {len(imgs)}")


✅ Env OK
📷 images in /datadrive/AutoPBR/photos_canary: 1154


## SAM3 (segmentation)


In [ ]:
sam3 = cfg.get("sam3", {})
if sam3.get("enabled", True):
    script = sam3.get("script", "sam3_v3.py")

    print(sam3)

    # pass only known scalar args if provided
    args = {}
    for k in ["out_dir", "score_thr", "mask_thr", "images_dir", "pattern"]:
        if k in sam3:
            kk = k.replace("_", "-")
            args[kk] = sam3[k]

    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ SAM3 disabled in config")


## Proxy semantic check


In [10]:
proxy = cfg.get("proxy_check", {})
if proxy.get("enabled", True):
    script = proxy.get("script", "proxy_semantic_check.py")
    args = {k: v for k, v in proxy.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ proxy_check disabled in config")



▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/proxy_semantic_check.py --out_dir data_output_canary/sam3_instances --manifest data_output_canary/sam3_instances/manifest.json
🔌 device=cuda
🅰️ Proxy A (SegFormer): nvidia/segformer-b5-finetuned-ade-640-640
🅱️ Proxy B (OneFormer): shi-labs/oneformer_ade20k_swin_large


Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 7699.98it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]             
The image processor of type `OneFormerImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 826/826 [00:00<00:00, 7293.61it/s, Materializing param=model.transformer_module.queries_embedder.weight]                                                       


✅ Done
📄 data_output_canary/sam3_instances/proxy_global_iou.csv
📄 data_output_canary/sam3_instances/proxy_per_building_iou.csv


## Nemotron structured


In [4]:
import os
from openai import OpenAI

# 1. Try to see what proxies your server is using right now
print("Current Proxy Settings:")
for k, v in os.environ.items():
    if 'proxy' in k.lower():
        print(f"  {k} = {v}")

api_key = os.getenv("NVIDIA_API_KEY")
if not api_key:
    print("\n❌ ERROR: NVIDIA_API_KEY is not set!")
    exit()

print("\n🔄 Attempting to connect to NVIDIA API...")

try:
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)
    
    # Send a tiny ping to the API
    response = client.chat.completions.create(
        model="nvidia/llama-3.1-nemotron-nano-vl-8b-v1",
        messages=[{"role": "user", "content": "Reply with 'API is working!'"}]
    )
    print("\n✅ SUCCESS! The API replied:")
    print(response.choices[0].message.content)
    
except Exception as e:
    print(f"\n❌ CONNECTION FAILED: {type(e).__name__}")
    print(str(e))

Current Proxy Settings:

🔄 Attempting to connect to NVIDIA API...

✅ SUCCESS! The API replied:
API is working!


In [4]:
ns = cfg.get("nemotron_structured", {})
if ns.get("enabled", True):
    script = ns.get("script", "nemotron_building_priors.py")
    args = {k: v for k, v in ns.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ nemotron_structured disabled in config")



▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/nemotron_building_priors.py --model nvidia/llama-3.1-nemotron-nano-vl-8b-v1 --canon_mode soft --per_building --skip_far_buildings --manifest data_output_canary/sam3_instances/manifest.json --masked_dir data_output_canary/masked_rgb --out_json data_output_canary/materials_full_filtered.json --min_cov 1.0 --min_cov_roof 1.5 --min_cov_building 0.5 --min_cov_roof_building 0.5 --roof_conf_threshold 0.6 --crop_pad 24 --min_crop_side 64 --max_retries 3 --retry_backoff 0.8
🔄 Found existing output file at data_output_canary/materials_full_filtered.json. Loading progress...
[1/1154] ⏭️ 1001053164315213 (Already processed, skipping)
[2/1154] ⏭️ 1003889610144908 (Already processed, skipping)
[3/1154] ⏭️ 1010222730367275 (Already processed, skipping)
[4/1154] ⏭️ 1011809876022469 (Already processed, skipping)
[5/1154] ⏭️ 101398269500832 (Already processed, skipping)
[6/1154] ⏭️ 1017514498889795 (Already processed, skipping)
[7/115

## Baseline full-image


In [7]:
nbaseline = cfg.get("nemotron_baseline", {})
if nbaseline.get("enabled", False):
    script = nbaseline.get("script", "nemotron_fullimage.py")
    args = {k: v for k, v in nbaseline.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx)
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ baseline disabled in config")



▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/nemotron_fullimage.py --model nvidia/llama-3.1-nemotron-nano-vl-8b-v1 --prompt_type two_pass_json_review --source_manifest data_output_canary/sam3_instances/manifest.json --output_json data_output_canary/baseline_full_image.json
[1/1154] 📸 image_1
   road: concrete (conf=0.80)
   wall: metal (conf=0.50)
   roof: metal (conf=0.50)
   door: unknown (conf=0.00)
   window: unknown (conf=0.00)
[2/1154] 📸 image_2
   road: asphalt (conf=0.80)
   wall: other:not_visible (conf=1.00)
   roof: other:not_visible (conf=1.00)
   door: unknown (conf=0.00)
   window: unknown (conf=0.00)
[3/1154] 📸 image_3
   road: asphalt (conf=1.00)
   wall: brick (conf=1.00)
   roof: other:not_visible (conf=1.00)
   door: unknown (conf=0.00)
   window: unknown (conf=0.00)
[4/1154] 📸 image_4
   road: other:not_visible (conf=1.00)
   wall: metal (conf=0.80)
   roof: other:not_visible (conf=1.00)
   door: unknown (conf=0.00)
   window: unknown (conf=

## Validation (optional)

In [ ]:
val = cfg.get("validation", {})
if val.get("enabled", False):
    script = val.get("script", "validator.py")
    args = {k: v for k, v in val.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ validation disabled in config")


## Quick check outputs


In [ ]:
out_candidates = [
    REPO_ROOT / out_root / "materials_full_filtered.json",
    REPO_ROOT / out_root / "baseline_full_image.json",
    REPO_ROOT / out_root / "sam3_instances" / "manifest.json",
    REPO_ROOT / out_root / "validation_results" / "report.csv",
]
for p in out_candidates:
    print(("✅" if p.exists() else "❌"), p)


## Annotations Visualization and Ground Truth Generation

In [ ]:
ann = cfg.get("annotations", {})
if ann.get("enabled", False):
    script = ann.get("script", "gui_annotator.py")
    cmd = f"{sys.executable} {q(REPO_ROOT / script)}".strip()
    run(cmd)
else:
    print("⏭ annotations visualization disabled in config")

## Detection Metrics Evaluation

In [17]:
met = cfg.get("det_metrics", {})
if met.get("enabled", False):
    script = met.get("script", "evaluate_detection.py")
    args = {k: v for k, v in met.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ metrics evaluation disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/evaluate_detection.py --baseline-json data_output_canary/baseline_full_image.json --structured-json data_output_canary/materials_full_filtered.json --gt-json data_output_canary/ground_truth_eval.json --output-json data_output_canary/recognition_evaluation_report.json
⏳ Loading JSON files...
🔍 Evaluating Images...

📊 CLASSIFICATION METRICS (Baseline vs. Global vs. GT)

🏗️  --- WALL (Total Valid Objects: 3366) ---
  [MACRO AVERAGES]
    BASELINE -> Acc: 0.00 | Prec: 0.00 | Rec: 0.00 | F1: 0.00
    GLOBAL   -> Acc: 0.67 | Prec: 0.32 | Rec: 0.33 | F1: 0.32

  [PER-CLASS RESULTS]
    🔸 brick (Support: 1016.0)
        Baseline -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
        Global   -> Prec: 0.62 | Rec: 0.82 | F1: 0.71
    🔸 concrete (Support: 1803.0)
        Baseline -> Prec: 0.00 | Rec: 0.00 | F1: 0.00
        Global   -> Prec: 0.76 | Rec: 0.63 | F1: 0.69
    🔸 glass (Support: 498.0)
        Baseline -> Prec: 0.00 | Rec: 0.

## Material Download

In [5]:
cmd = f"{sys.executable} {q(REPO_ROOT / "download_materials.py")}".strip()
run(cmd)


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/download_materials.py
🌐 Fetching texture list from Poly Haven API...
✅ Found 744 textures. Starting RGB download...

📂 Category: CONCRETE
   ⬇️ Downloaded RGB: anti_slip_concrete.jpg
   ⬇️ Downloaded RGB: asphalt_floor.jpg
   ⬇️ Downloaded RGB: brushed_concrete.jpg
   ⬇️ Downloaded RGB: brushed_concrete_03.jpg
   ⬇️ Downloaded RGB: brushed_concrete_2.jpg
   ⬇️ Downloaded RGB: climbing_wall_02.jpg
   ⬇️ Downloaded RGB: concrete.jpg
   ⬇️ Downloaded RGB: concrete_block_wall.jpg
   ⬇️ Downloaded RGB: concrete_block_wall_02.jpg
   ⬇️ Downloaded RGB: concrete_debris.jpg
   ⬇️ Downloaded RGB: concrete_floor.jpg
   ⬇️ Downloaded RGB: concrete_floor_01.jpg
   ⬇️ Downloaded RGB: concrete_floor_02.jpg
   ⬇️ Downloaded RGB: concrete_floor_damaged_01.jpg
   ⬇️ Downloaded RGB: concrete_floor_painted.jpg
📂 Category: ASPHALT
   ⬇️ Downloaded RGB: aerial_asphalt_01.jpg
   ⬇️ Downloaded RGB: asphalt_01.jpg
   ⬇️ Downloaded RGB: aspha

In [7]:
cmd = f"{sys.executable} {q(REPO_ROOT / "tint_material.py")}".strip()
run(cmd)


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/tint_material.py
🎨 Generating color variations + preserving originals...

🔄 Processing CONCRETE...
🔄 Processing ASPHALT...
🔄 Processing WOOD...
🔄 Processing METAL...
🔄 Processing BRICK...
🔄 Processing PLASTER...
🔄 Processing TILE...
🔄 Processing SHINGLES...

🎉 Done! Generated 1180 colorized variations and preserved originals in 'colored_material_bank/'.


## Material Replacement with Natural Language

In [23]:
nmat = cfg.get("nlp_material_replacement", {})
if nmat.get("enabled", False):
    user_input = input("Enter your edit prompt (e.g., 'Change the wall of the biggest building on the right to brown brick'): ")
    script = nmat.get("script", "generation_test.py")
    args = {k: v for k, v in nmat.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} --prompt \"{user_input}\" {cli}".strip()
    run(cmd)
else:
    print("⏭ generation test disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/generation_test.py --prompt "Change the wall of the biggest building on the right to brown brick" --json data_output_canary/ground_truth_eval.json

⏳ Loading Models...


Loading weights: 100%|██████████| 414/414 [00:00<00:00, 6746.61it/s, Materializing param=neck.reassemble_stage.readout_projects.3.0.weight]                          
The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 520/520 [00:00<00:00, 2513.35it/s, Materializing param=visual_projection.weight]                                
CLIPVisionModelWithProjection LOAD REPORT from: models/image_encoder
Key                

🖼️ Attempting Image ID: 1003889610144908...
🧠 Asking AI to interpret prompt...
✅ Final Parsed Intent: {
  "target_instance": "building_06",
  "target_structure": "wall",
  "new_color": "brown",
  "new_material": "brick"
}
✅ Processing 1003889610144908 -> building_06...
  -> Running Naive Edit (No Segmentation)...


100%|██████████| 27/27 [00:10<00:00,  2.69it/s]
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/diffusers/pipelines/controlnet/pipeline_controlnet_inpaint_sd_xl.py:1131: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


  -> Running Proposed Edit (Segmented)...


100%|██████████| 27/27 [00:10<00:00,  2.68it/s]


  -> Running Improved Edit (Lighting Match + Compositing)...


100%|██████████| 25/25 [00:09<00:00,  2.67it/s]



✅ Success! Generation Complete. Results in nlp_results/


## Generation Metrics Evaluation

In [29]:
gen = cfg.get("gen_bench", {})
if nmat.get("enabled", False):
    script = gen.get("script", "generate_benchmark_set.py")
    args = {k: v for k, v in gen.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ benchmark generation disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/generate_benchmark_set.py --benchmark-json data_output_canary/benchmark_dataset.json --output-dir data_output_canary/benchmark_results --gt-json data_output_canary/ground_truth_eval.json
📖 Loading Ground Truth data...
🎨 Found 53 realistic, plausible material styles in the bank.

⏳ Loading Models (ControlNet & IPAdapter)...


Loading weights: 100%|██████████| 414/414 [00:00<00:00, 8501.76it/s, Materializing param=neck.reassemble_stage.readout_projects.3.0.weight]                          
The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
/usr/local/miniconda3.8/envs/SasyEnv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
Loading weights: 100%|██████████| 517/517 [00:00<00:00, 2513.19it/s, Materializing param=text_projection.weight]

Loading weights: 100%|██████████| 520/520 [00:00<00:00, 835.48it/s, Materializing param=visual_projection.weight]        

🚀 Models Loaded.

🎉 Benchmark Generation Complete! Saved 300 pairs to data_output_canary/benchmark_dataset.json


In [33]:
genm = cfg.get("gen_metrics", {})
if met.get("enabled", False):
    script = genm.get("script", "evaluate_generation.py")
    args = {k: v for k, v in genm.items() if k not in ["enabled", "script"]}
    cli = args_to_cli(args, ctx) if args else ""
    cmd = f"{sys.executable} {q(REPO_ROOT / script)} {cli}".strip()
    run(cmd)
else:
    print("⏭ generation metrics evaluation disabled in config")


▶ /usr/local/miniconda3.8/envs/SasyEnv/bin/python /datadrive/AutoPBR/evaluate_generation.py --benchmark-json data_output_canary/benchmark_dataset.json
📖 Loading Benchmark Data...

⏳ Loading CLIP Model for Text-Image Alignment...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 9578.84it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Initializing FID Metrics...

🚀 Starting Evaluation...


Evaluating Images: 100%|██████████| 300/300 [02:53<00:00,  1.73it/s]



⏳ Computing final FID scores (this takes a few seconds)...

 📊 EVALUATION RESULTS: ORIGINAL vs NAIVE vs PROPOSED vs IMPROVED
Metric               | Orig (Baseline)    | Naive (Global)   | Proposed (Masked)  | Improved (Comp.)  
---------------------------------------------------------------------------------------------------------
Bg SSIM (↑)          |       1.0000       |      0.3419      |       0.8479       |       0.9978      
Global CLIP (↑)      |       18.67        |      26.31       |       20.77        |       19.88       
Masked CLIP (↑)      |       24.33        |      25.22       |       24.54        |       24.15       
FID Realism (↓)      |        0.00        |      174.36      |       22.99        |       14.82       

✅ Detailed evaluation saved to generation_evaluation_report.json
